# 10 V6+V10 ensemble + hflip TTA — submission

**Inference + submission.** Produces `submission.csv` from the configuration `src/09` confirmed beats
V6 on the comparable val metric: full-image **V6+V10 class-wise NMS ensemble + manual hflip TTA**
(Ensemble+TTA = +0.0082 vs V6 on val, large classes up, no regression). Full-image inference — NOT
the tiled path in `src/02` (that was the failed V13 pipeline).

**Confidence floor is chosen on val, not guessed.** Section 4 reruns the exact Ensemble+TTA pipeline
on the val split, scores it with the `src/09` comparable Mask mAP across a grid of floors, and picks
the **highest floor within the 0.003 noise band of the best** — the cleanest/smallest file that does
not cost mAP (a hard threshold can only truncate the PR curve, so for an mAP-scored leaderboard you
want a low-but-not-noisy floor). Set `MANUAL_SUBMIT_CONF` to override; set `RUN_CONF_SWEEP=False` (or
omit the training dataset) to fall back to `DEFAULT_SUBMIT_CONF`.

**Inputs (Kaggle):**
- the competition dataset (test images + `sample_submission.csv`);
- the **V6** (`version6...best.pt`) and **V10** (`version10...best.pt`) detectors;
- *(for the floor sweep)* the training dataset with the val split (`yolo_seg_train.yaml`) — the same
  one `src/09` used. Without it the sweep is skipped and `DEFAULT_SUBMIT_CONF` is used.

**Submission format (same as src/02):** `id,patient_id,class_id,confidence,poly` (full-image
normalized polygons).

## 1. Environment setup

Kaggle submission notebooks often run offline. Installs `ultralytics` from an offline wheel dataset
if present (set `MANUAL_WHEELS_DIR` if auto-detection misses it); set `ALLOW_INTERNET_INSTALL=True`
only when testing with Internet on.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

MANUAL_WHEELS_DIR = None
ALLOW_INTERNET_INSTALL = True   # this competition allows Internet; install ultralytics from PyPI if absent

def _pkg(n): return importlib.util.find_spec(n) is not None
def _wheel_dirs(root=Path("/kaggle/input")):
    if not root.exists(): return []
    dirs = sorted({p.parent for p in root.rglob("*.whl")})
    return sorted(dirs, key=lambda d: any("ultralytics" in p.name.lower() for p in d.glob("*.whl")), reverse=True)

if not _pkg("ultralytics"):
    wd = ([Path(MANUAL_WHEELS_DIR)] if MANUAL_WHEELS_DIR else []) + _wheel_dirs()
    seen = set(); wd = [d for d in wd if not (str(d) in seen or seen.add(str(d)))]
    if wd:
        cmd = [sys.executable, "-m", "pip", "install", "--no-index"]
        for d in wd: cmd += ["--find-links", str(d)]
        cmd.append("ultralytics"); subprocess.check_call(cmd)
    elif ALLOW_INTERNET_INSTALL:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    else:
        raise RuntimeError("ultralytics missing and no offline wheels found. Add wheels, set "
                           "MANUAL_WHEELS_DIR, or enable Internet for testing.")
else:
    print("ultralytics present")

In [ ]:
import gc, json
import numpy as np
import pandas as pd
import cv2
import yaml
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO

cv2.setNumThreads(0)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| cuda:", torch.cuda.is_available())

# --- Pipeline (must match src/09's winning Ensemble+TTA) ---
IMG_SIZE     = 768
CAPTURE_CONF = 0.001      # capture low; the real floor is chosen on val (section 4)
PRED_NMS_IOU = 0.70       # per-model predict-time NMS
MAX_DET      = 300
ENS_NMS_IOU  = 0.60       # cross-model / cross-view class-wise NMS merge
RETINA_MASKS = False      # memory-safe
USE_HALF     = True

# --- Confidence floor selection ---
RUN_CONF_SWEEP     = True
MANUAL_SUBMIT_CONF = None                                  # set a float to override the sweep
DEFAULT_SUBMIT_CONF = 0.05                                 # fallback if the sweep is skipped
CONF_GRID    = [0.001, 0.01, 0.02, 0.05, 0.10, 0.15, 0.25, 0.40]
NOISE_BAND   = 0.003                                       # pick the highest floor within this of best
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)                # mask-AP thresholds (src/03/04/05/09)
print("imgsz", IMG_SIZE, "| ens NMS IoU", ENS_NMS_IOU, "| conf grid", CONF_GRID)

## 2. Load the V6 and V10 detectors (auto-detect by filename)

In [ ]:
MANUAL_V6_PATH  = None
MANUAL_V10_PATH = None

def find_pt(include, exclude=()):
    cands = []
    for p in Path("/kaggle/input").rglob("*.pt"):
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if p.name.lower().endswith("best.pt") else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

v6  = find_pt(["version6", "v6"],   exclude=["stage2", "version10", "version13", "version14", "version15"])
v10 = find_pt(["version10", "v10"], exclude=["stage2", "version6"])
V6_PATH  = Path(MANUAL_V6_PATH)  if MANUAL_V6_PATH  else (v6[0]  if v6  else None)
V10_PATH = Path(MANUAL_V10_PATH) if MANUAL_V10_PATH else (v10[0] if v10 else None)
print("V6 :", V6_PATH); print("V10:", V10_PATH)
if V6_PATH is None or not V6_PATH.exists():
    raise FileNotFoundError("No V6 detector found. Add version6_...best.pt or set MANUAL_V6_PATH.")
if V10_PATH is None or not V10_PATH.exists():
    raise FileNotFoundError("No V10 detector found. Add version10_...best.pt or set MANUAL_V10_PATH.")
detectors = {"V6": YOLO(str(V6_PATH)), "V10": YOLO(str(V10_PATH))}

## 3. The Ensemble+TTA pipeline (shared by the val sweep and test inference)

`ensemble_predict(img)` = V6 and V10 each on the image **and its hflip** (detections mirrored back),
pooled and merged by class-wise NMS at `ENS_NMS_IOU`. Returns full-image-normalized merged
detections `[{cls, conf, poly}]` — identical logic to src/09's `Ensemble+TTA` variant.

In [ ]:
def predict_dets(det, img):
    res = det.predict(source=img, imgsz=IMG_SIZE, conf=CAPTURE_CONF, iou=PRED_NMS_IOU,
                      max_det=MAX_DET, device=DEVICE, task="segment", retina_masks=RETINA_MASKS,
                      half=(USE_HALF and DEVICE != "cpu"), verbose=False, save=False)[0]
    out = []
    if res.masks is not None and res.boxes is not None and len(res.boxes) > 0:
        xyxy = res.boxes.xyxy.cpu().numpy(); conf = res.boxes.conf.cpu().numpy()
        cls = res.boxes.cls.cpu().numpy().astype(int); xyn = res.masks.xyn
        for i in range(min(len(xyxy), len(xyn))):
            poly = np.asarray(xyn[i], dtype=np.float64)
            if poly.ndim != 2 or len(poly) < 3: continue
            x0, y0, x1, y1 = [float(v) for v in xyxy[i]]
            out.append({"cls": int(cls[i]), "conf": float(conf[i]), "box": (x0, y0, x1, y1), "poly": poly})
    return out

def hflip_back(dets, w):
    out = []
    for d in dets:
        x0, y0, x1, y1 = d["box"]
        p = d["poly"].copy(); p[:, 0] = 1.0 - p[:, 0]
        out.append({"cls": d["cls"], "conf": d["conf"], "box": (w - x1, y0, w - x0, y1), "poly": p})
    return out

def box_iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1]); ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0); inter = iw * ih
    if inter <= 0: return 0.0
    aa = (a[2]-a[0])*(a[3]-a[1]); bb = (b[2]-b[0])*(b[3]-b[1])
    return inter / (aa + bb - inter)

def nms_merge(dets, iou_thr):
    keep = []
    for c in set(d["cls"] for d in dets):
        cd = sorted([d for d in dets if d["cls"] == c], key=lambda x: x["conf"], reverse=True)
        sel = []
        for d in cd:
            if all(box_iou(d["box"], s["box"]) < iou_thr for s in sel): sel.append(d)
        keep.extend(sel)
    return keep

def ensemble_predict(img):
    h, w = img.shape[:2]
    img_f = cv2.flip(img, 1)
    pool = []
    for det in detectors.values():
        pool += predict_dets(det, img)
        pool += hflip_back(predict_dets(det, img_f), w)
    return nms_merge(pool, ENS_NMS_IOU)

def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

## 4. Choose the confidence floor on the val split

Build the Ensemble+TTA merged detections per val image once, then for each floor score the comparable
Mask mAP and pick the **highest floor within `NOISE_BAND` of the best**. Skipped if `RUN_CONF_SWEEP`
is off or the training yaml (val split) is not attached → `DEFAULT_SUBMIT_CONF`.

In [ ]:
def poly_to_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:,0].min()))); gy0 = max(0, int(np.floor(pts[:,1].min())))
    gx1 = min(w, int(np.ceil(pts[:,0].max())));  gy1 = min(h, int(np.ceil(pts[:,1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8)
    pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0: return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3])
    inter = 0
    if ix1 > ix0 and iy1 > iy0:
        pc = pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc = gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

def ap_101(confs, tps, npos):
    if npos == 0: return float("nan")
    if not confs: return 0.0
    o = np.argsort(-np.asarray(confs)); tps = np.asarray(tps)[o]
    tp_c = np.cumsum(tps); fp_c = np.cumsum(1 - tps)
    rec = tp_c / npos; prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return sum((prec[rec >= r].max() if np.any(rec >= r) else 0.0) for r in np.linspace(0,1,101)) / 101.0

def _load_val_gt():
    yc = list(Path("/kaggle/input").rglob("yolo_seg_train.yaml"))
    if not yc: return None, None, None
    dcfg = yaml.safe_load(open(yc[0], encoding="utf-8"))
    nm = dcfg.get("names");  nm = [nm[k] for k in sorted(nm)] if isinstance(nm, dict) else nm
    root = yc[0].parent
    yroot = Path(dcfg["path"]) if dcfg.get("path") else root
    if not yroot.is_absolute(): yroot = root / yroot
    def resolve(v):
        p = Path(v)
        if p.is_absolute(): return p
        for c in (yroot / p, root / p):
            if c.exists(): return c
        return yroot / p
    vimg = resolve(dcfg.get("val", dcfg.get("valid")))
    parts = list(Path(vimg).parts)
    if "images" in parts:
        i = len(parts)-1-parts[::-1].index("images"); parts[i] = "labels"; ldir = Path(*parts)
    else:
        ldir = Path(vimg).parent / "labels"
    exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
    objs = {}
    for ip in sorted(p for p in Path(vimg).iterdir() if p.suffix.lower() in exts):
        lp = ldir / (ip.stem + ".txt"); o = []
        if lp.exists():
            for line in lp.read_text().splitlines():
                t = line.strip().split()
                if len(t) < 7: continue
                try: c = int(float(t[0])); co = [float(x) for x in t[1:]]
                except ValueError: continue
                if len(co) % 2: co = co[:-1]
                pl = np.asarray(co, dtype=np.float64).reshape(-1, 2)
                if len(pl) >= 3: o.append((c, pl))
        objs[str(ip)] = o
    return nm, objs, len(nm)

SUBMIT_CONF = None
if MANUAL_SUBMIT_CONF is not None:
    SUBMIT_CONF = float(MANUAL_SUBMIT_CONF)
    print("Using MANUAL_SUBMIT_CONF =", SUBMIT_CONF)
elif not RUN_CONF_SWEEP:
    SUBMIT_CONF = DEFAULT_SUBMIT_CONF
    print("Sweep off -> DEFAULT_SUBMIT_CONF =", SUBMIT_CONF)
else:
    names, val_objs, num_classes = _load_val_gt()
    if val_objs is None:
        SUBMIT_CONF = DEFAULT_SUBMIT_CONF
        print("No yolo_seg_train.yaml found -> DEFAULT_SUBMIT_CONF =", SUBMIT_CONF)
    else:
        n_gt = np.zeros(num_classes, dtype=int)
        for o in val_objs.values():
            for c, _ in o: n_gt[c] += 1
        gt_local = {ip: [(c, *poly_to_local_mask(p, *cv2.imread(ip).shape[1::-1])) for c, p in o]
                    for ip, o in val_objs.items()}
        val_merged = {}
        for ip in tqdm(val_objs, desc="val Ensemble+TTA"):
            img = cv2.imread(ip, cv2.IMREAD_COLOR)
            val_merged[ip] = (ensemble_predict(img), img.shape[1], img.shape[0])
            release_memory()
        def score_floor(floor):
            records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
            for ip in val_objs:
                gts = gt_local[ip]; dets, w, h = val_merged[ip]
                pls = [(d["cls"], d["conf"], *poly_to_local_mask(d["poly"], w, h))
                       for d in dets if d["conf"] >= floor]
                for ti, thr in enumerate(IOU_THRESHOLDS):
                    order = sorted(range(len(pls)), key=lambda i: pls[i][1], reverse=True)
                    used = [False]*len(gts)
                    for pi in order:
                        pc, sc, pbox, pm = pls[pi]; best, bg = 0.0, -1
                        for gi, (gc, gbox, gm) in enumerate(gts):
                            if used[gi] or gc != pc: continue
                            v = iou_local(pbox, pm, gbox, gm)
                            if v > best: best, bg = v, gi
                        tp = 1 if (bg >= 0 and best >= thr) else 0
                        if tp: used[bg] = True
                        records[(pc, ti)].append((sc, tp))
            pac = np.full(num_classes, np.nan)
            for c in range(num_classes):
                if n_gt[c] == 0: continue
                pac[c] = np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                            [r[1] for r in records[(c,ti)]], n_gt[c])
                                     for ti in range(len(IOU_THRESHOLDS))])
            return float(np.nanmean(pac[~np.isnan(pac)]))
        sweep = {f: score_floor(f) for f in CONF_GRID}
        best = max(sweep.values())
        SUBMIT_CONF = max(f for f, m in sweep.items() if m >= best - NOISE_BAND)
        print(f"{'floor':>8s}  {'val mAP':>8s}")
        for f in CONF_GRID:
            mark = "  <- best" if sweep[f] == best else ("  * chosen" if f == SUBMIT_CONF else "")
            print(f"{f:>8.3f}  {sweep[f]:>8.4f}{mark}")
        print(f"\nChosen SUBMIT_CONF = {SUBMIT_CONF}  (highest floor within {NOISE_BAND} of best {best:.4f})")

## 5. Locate test images + sample submission (from the competition dataset)

In [ ]:
def find_sample_submission():
    for p in [Path("/kaggle/input/competitions/alpha-dent/sample_submission.csv"),
              Path("/kaggle/input/alpha-dent/sample_submission.csv")]:
        if p.exists(): return p
    cands = sorted(Path("/kaggle/input").rglob("sample_submission.csv"))
    if not cands: raise FileNotFoundError("sample_submission.csv not found under /kaggle/input.")
    return sorted(cands, key=lambda p: ("alpha" in str(p).lower(), "dent" in str(p).lower()), reverse=True)[0]

SAMPLE_SUBMISSION_PATH = find_sample_submission()
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
expected_columns = sample_submission.columns.tolist()
print("sample submission:", SAMPLE_SUBMISSION_PATH, "| columns:", expected_columns)
for col in ["id", "patient_id", "class_id", "confidence", "poly"]:
    assert col in expected_columns, f"missing column {col}"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}
all_imgs = sorted(p for p in Path("/kaggle/input").rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)
pids = sorted(sample_submission["patient_id"].astype(str).unique())
by_stem = {}
for p in all_imgs: by_stem.setdefault(p.stem, p)
matched = [by_stem[pid] for pid in pids if pid in by_stem]
if matched:
    test_image_paths = sorted(matched, key=lambda p: p.stem)
    print("matched test images from sample_submission:", len(test_image_paths))
else:
    test_image_paths = sorted((p for p in all_imgs if "test" in str(p).lower()), key=lambda p: p.stem)
    print("fallback test images (path contains 'test'):", len(test_image_paths))
if not test_image_paths:
    raise FileNotFoundError("No test images found under /kaggle/input.")

## 6. Run test inference + build `submission.csv`

For each test image: `ensemble_predict` (full-image, V6+V10, orig+hflip, class-wise NMS) → keep
detections with `conf >= SUBMIT_CONF` → emit `id,patient_id,class_id,confidence,poly` (full-image
normalized poly). No tiling/untiling — polygons are already full-image.

In [ ]:
def polygon_to_string(poly):
    poly = np.clip(np.asarray(poly, dtype=np.float32), 0.0, 1.0).reshape(-1)
    return " ".join(f"{v:.6f}" for v in poly)

def validate_poly_string(poly):
    try: vals = [float(x) for x in str(poly).split()]
    except Exception: return False
    return len(vals) >= 6 and len(vals) % 2 == 0 and min(vals) >= 0.0 and max(vals) <= 1.0

assert SUBMIT_CONF is not None, "SUBMIT_CONF was not set."
print("Submitting at conf >=", SUBMIT_CONF)
rows = []
for ip in tqdm(test_image_paths, desc="test Ensemble+TTA"):
    img = cv2.imread(str(ip), cv2.IMREAD_COLOR)
    if img is None:
        print("WARNING unreadable, skipping:", ip.name); continue
    for d in ensemble_predict(img):
        if d["conf"] < SUBMIT_CONF: continue
        if not (d["poly"].ndim == 2 and len(d["poly"]) >= 3): continue
        rows.append({"patient_id": ip.stem, "class_id": int(d["cls"]),
                     "confidence": float(d["conf"]), "poly": polygon_to_string(d["poly"])})
    release_memory()

submission_df = pd.DataFrame(rows, columns=["patient_id", "class_id", "confidence", "poly"])
submission_df.insert(0, "id", np.arange(len(submission_df), dtype=int))
submission_df = submission_df[expected_columns]
submission_df["id"] = submission_df["id"].astype(int)
submission_df["patient_id"] = submission_df["patient_id"].astype(str)
submission_df["class_id"] = submission_df["class_id"].astype(int)
submission_df["confidence"] = submission_df["confidence"].astype(float)
submission_df["poly"] = submission_df["poly"].astype(str)
print("submission shape:", submission_df.shape)
display(submission_df.head())

## 7. Sanity checks + save

In [ ]:
assert submission_df.columns.tolist() == expected_columns, "column order != sample_submission"
if len(submission_df) == 0:
    print("WARNING: zero rows — lower SUBMIT_CONF.")
else:
    assert submission_df["class_id"].between(0, 8).all(), "class_id out of [0,8]"
    assert submission_df["confidence"].between(0, 1).all(), "confidence out of [0,1]"
    vr = submission_df["poly"].head(1000).apply(validate_poly_string).mean()
    print("polygon validity (first 1000):", round(float(vr), 4))
    assert vr > 0.99, "some polygon strings invalid"

SUBMISSION_PATH = Path("/kaggle/working/submission.csv")
submission_df.to_csv(SUBMISSION_PATH, index=False)
print("saved:", SUBMISSION_PATH)
saved = pd.read_csv(SUBMISSION_PATH)
print("saved shape:", saved.shape)

if len(submission_df) > 0:
    print("\npredictions per class:")
    print(submission_df["class_id"].value_counts().sort_index().to_string())
    cpi = submission_df.groupby("patient_id").size()
    print("\npredictions per image: mean %.1f  min %d  max %d" % (cpi.mean(), cpi.min(), cpi.max()))

## 8. Notes

- **What was submitted:** full-image V6+V10 ensemble + hflip TTA, class-wise NMS at `ENS_NMS_IOU`,
  floor `SUBMIT_CONF` (chosen on val in section 4 — `MANUAL_SUBMIT_CONF` overrides).
- **If the LB gain is smaller than the +0.0082 val delta:** the val anchor (0.205) sits below the
  documented 0.234 (a known metric-impl gap), so treat the *direction* as the signal. The ensemble is
  zero-training, so a confirmed val gain should still transfer, just modestly.
- **Cost:** 4 forward passes/image (V6/V10 × orig/hflip). If you must cut cost, `src/09` showed
  neither half alone clears noise — so dropping to a cheaper variant likely forfeits the gain.
- **Knobs:** `ENS_NMS_IOU` (lower = merge harder), add TTA views (vflip / multi-scale), or
  confidence-weight the two models before pooling — each a single-variable follow-up.